## Feature Selection Strategy Implementation (Sections 3-4)

This notebook implements the feature-selection pipeline from the plan: broad univariate filtering, multicollinearity reduction, RFECV-based ordering, and the downstream modeling workflow.

### Covered Workflow
- Stage 1: Broad univariate filtering (500 -> ~80)
- Stage 2: Multicollinearity and L1 regularization (80 -> ~25)
- Stage 3: RFECV ordering of Stage 2 features for top-k model comparisons
- Stage 4: Model comparison, threshold tuning, and final test ranking

### Included Components
- Data loading and sanity checks
- Custom business scorer aligned with project metric
- Stage-by-stage feature reduction and validation
- RFECV ranking of Stage 2 features
- Evaluation of models on multiple feature-set sizes
- OOF probability curves for threshold tuning
- Final feature locking, test ranking, and export artifacts

Markdown is concise by design; implementation is performed in the code cells.

## Key Parameters

| Parameter | Default | Description |
|---|---|---|
| custom_scorer | Required | Scoring function: scorer(estimator, X, y) -> float |
| stage1_n_features | 80 | Target features after Stage 1 |
| stage2_n_features | None | Optional strict target after Stage 2 |
| stage2_n_features_band | (20, 30) | Preferred approximate Stage 2 band (data-driven final count) |
| correlation_threshold | 0.85 | Correlation pruning threshold |
| vif_threshold | 5.0 | VIF filtering threshold |
| variance_threshold | 0.01 | Variance threshold (Stage 1) |
| rfecv_step | 1 | Step size for RFECV feature elimination |
| rfecv_cv | 5 | Cross-validation folds used by RFECV |
| rfecv_n_jobs | -1 | Parallel jobs for RFECV fitting |
| candidate_feature_sizes | (1, 3, 5, 8, 10, 15, 20, all) | Top-k feature subsets evaluated in Stage 4 |
| modeling_cv_folds | 5 | Cross-validation folds used for model comparison |
| max_test_targets | 1000 | Maximum ranked test samples to retain |
| include_xgboost | False | Optional XGBoost factory when the dependency is available |
| probability_threshold_floor | 1 / 3 | Break-even probability floor for positive predictions |
| random_state | 42 | Random seed |
| verbose | 1 | Verbosity level (0=silent, 1=progress) |

## Usage Notes

All runnable examples are implemented directly in this notebook as executable code cells.
This section is kept intentionally concise to avoid duplicated code in markdown.

## Performance Expectations

1. Reduce dimensionality from 500 to approximately 20-30 features.
2. Retain high-signal predictors.
3. Maximize business score by removing low-value features.
4. Finish in under ~20 minutes on a standard laptop with 5-fold CV.

Typical timing:
- Stage 1: 10-15 minutes
- Stage 2: 5-10 minutes
- Stage 3: depends on RFECV cost across the Stage 2 feature set
- Stage 4: depends on model comparison and OOF probability curves

## Troubleshooting

- If too many features remain, tighten the Stage 2 band or increase regularization.
- If too few features remain, widen the Stage 2 band or relax regularization.
- If interpretability is weak, use permutation importance and remove low-contribution features.

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_selection import RFECV

# Add src to path
PROJECT_ROOT: Path = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [2]:
from cost_effective.dataset import (
    MulticollinearityFilter,
    UnivariateFeatureFilter,
    custom_scorer,
    get_classifier,
)

# Configuration
np.random.seed(42)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Constants
DATA_PATH: Path = PROJECT_ROOT / "data"
OUTPUTS_PATH: Path = PROJECT_ROOT / "outputs"

In [3]:
# Load raw data
X_all = pd.read_csv(DATA_PATH / "x_train.txt", sep=r"\s+", header=None, low_memory=False).apply(
    pd.to_numeric, errors="coerce"
)
y_raw = pd.to_numeric(
    pd.read_csv(DATA_PATH / "y_train.txt", sep=r"\s+", header=None).iloc[:, 0], errors="coerce"
)

# Keep only rows with valid binary target and aligned feature rows.
valid_mask = y_raw.notna() & y_raw.isin([0, 1])
X_all = X_all.loc[valid_mask].reset_index(drop=True)
y = y_raw.loc[valid_mask].astype(int).reset_index(drop=True)

n_features_all = X_all.shape[1]
X_all.columns = [f"var_{i}" for i in range(n_features_all)]
y.name = "target"

print(f"✓ Loaded raw data: {X_all.shape}")
print(f"  Class distribution: {y.value_counts().to_dict()}")

✓ Loaded raw data: (5000, 500)
  Class distribution: {0: 2512, 1: 2488}


In [ ]:
# Stage 1 and Stage 2
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*'penalty' was deprecated.*")
    warnings.filterwarnings("ignore", message=".*Inconsistent values: penalty=l1.*")

    print("=" * 70)
    print("STAGE 1: UNIVARIATE FILTERING (500 → 80)")
    print("=" * 70 + "\n")

    stage1_filter = UnivariateFeatureFilter(variance_threshold=0.01)
    X_stage1, stage1_features = stage1_filter.fit_transform(X_all, y, n_features=80)

    print()
    stage2_filter = MulticollinearityFilter(
        correlation_threshold=0.85,
        vif_threshold=5.0,
        random_state=42,
    )
    X_stage2, stage2_features = stage2_filter.fit_transform(
        X_stage1,
        y,
        n_features_target=None,
        n_features_band=(20, 30),
    )

print(f"✓ Stage 1 output: {len(stage1_features)} features")
print(f"✓ Stage 2 output: {len(stage2_features)} features")

STAGE 1: UNIVARIATE FILTERING (500 → 80)

[Stage 1a] Variance Threshold: 500 → 500 features
[Stage 1b] Mutual Information computed for 500 features
  Top 10 MI scores:
     feature  mi_score
10    var_10  0.029785
456  var_456  0.021367
470  var_470  0.020561
312  var_312  0.018987
159  var_159  0.018163
415  var_415  0.018163
175  var_175  0.018041
254  var_254  0.017258
4      var_4  0.017163
379  var_379  0.017061

[Stage 1c] LightGBM Importance computed
  Top 10 LGBM Importances:
     feature  lgbm_importance
214  var_214               94
379  var_379               94
341  var_341               81
254  var_254               73
190  var_190               71
116  var_116               67
159  var_159               64
226  var_226               61
389  var_389               60
482  var_482               58

[Stage 1 Output] Selected 80 features out of 500

STAGE 2: MULTICOLLINEARITY & L1 REGULARIZATION FILTERING

[Stage 2a] Correlation Pruning (threshold=0.85):
  High-correlation pair

In [ ]:
# Stage 3
print("=" * 70)
print("STAGE 3: RFECV WITH CUSTOM BUSINESS SCORER")
print("=" * 70)

# Configure RFECV with custom scorer
rfecv = RFECV(
    estimator=get_classifier(y=y),
    step=1,
    cv=5,
    scoring=custom_scorer,
    n_jobs=-1,
    verbose=0,
)

# Fit on ndarray and silence known sklearn feature-name warning from RFECV internals.
X_stage2_np = X_stage2.to_numpy()
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*X does not have valid feature names.*")
    rfecv.fit(X_stage2_np, y)

print("\n✓ RFECV completed")
print(f"  Ranking vector: {rfecv.ranking_}")

STAGE 3: RFECV WITH CUSTOM BUSINESS SCORER

✓ RFECV completed
  Ranking vector: [ 3  2 22  7  1 14  5 10 16 18 13 20 15 19  4  6 11  8 21 26 23  9 12 24
 25 17]


In [6]:
df = pd.DataFrame({
    "feature": X_stage2.columns,
    "ranking": rfecv.ranking_,
})
df.to_csv(OUTPUTS_PATH / "feature_selection_results.csv", index=False)

In [ ]:
from cost_effective.models import build_top_k_feature_sets, evaluate_feature_sets, rank_features

ranked_stage2_features = rank_features(list(X_stage2.columns), rfecv.ranking_)

candidate_feature_sizes = (1, 3, 5, 8, 10, 15, 20, len(stage2_features))
feature_set_candidates = build_top_k_feature_sets(
    ranked_stage2_features["feature"],
    sizes=candidate_feature_sizes,
)

stage3_feature_set_scores = evaluate_feature_sets(
    X_stage2,
    y,
    feature_set_candidates,
    estimator_factory=get_classifier,
    cv=5,
)

print(stage3_feature_set_scores.to_string(index=False))

## Modeling Strategy Implementation (Section 4)

This section evaluates multiple model families on ranked feature subsets, then uses out-of-fold probabilities to tune the final business score and rank test predictions.

In [ ]:
X_test = pd.read_csv(DATA_PATH / "x_test.txt", sep=r"\s+", header=None, low_memory=False).apply(
    pd.to_numeric, errors="coerce"
)
X_test.columns = X_all.columns
X_test = X_test.fillna(0.0)

print(f"✓ Loaded test data: {X_test.shape}")

In [ ]:
from cost_effective.models import (
    build_model_factories,
    build_profit_curve,
    compare_models_on_feature_sets,
    compute_oof_probabilities,
    fit_final_model_and_predict,
)

model_factories = build_model_factories(y)
model_comparison = compare_models_on_feature_sets(
    X_stage2,
    y,
    feature_set_candidates,
    estimator_factories=model_factories,
    cv=5,
)

best_model_row = model_comparison.iloc[0]
best_model_name = str(best_model_row["model_name"])
best_feature_set_name = str(best_model_row["feature_set_name"])
best_features = feature_set_candidates[best_feature_set_name]

print(model_comparison.to_string(index=False))
print()
print(f"Best model: {best_model_name}")
print(f"Best feature set: {best_feature_set_name} ({len(best_features)} features)")

oof_probabilities = compute_oof_probabilities(
    X_stage2[best_features],
    y,
    estimator_factory=model_factories[best_model_name],
    cv=5,
)
profit_curve = build_profit_curve(
    y,
    oof_probabilities,
    feature_count=len(best_features),
    max_targets=1000,
)

print(profit_curve.curve.head(10).to_string(index=False))
print()
print(
    f"Best OOF cutoff: k={profit_curve.best_k}, threshold={profit_curve.best_threshold:.4f}, "
    f"score={profit_curve.best_score:.2f}"
)

final_prediction_result = fit_final_model_and_predict(
    X_train=X_stage2,
    y_train=y,
    X_test=X_test,
    selected_features=best_features,
    estimator_factory=model_factories[best_model_name],
    max_targets=1000,
)

final_prediction_frame = pd.DataFrame({
    "rank": np.arange(1, len(final_prediction_result.ranked_test_indices) + 1),
    "sample_index": final_prediction_result.ranked_test_indices,
    "probability": final_prediction_result.probabilities[
        final_prediction_result.ranked_test_indices
    ],
})

final_prediction_frame.to_csv(OUTPUTS_PATH / "model_predictions.csv", index=False)
final_prediction_frame.head()